# 01 · Bronze — Extração das contratações do PNCP (Pará)

Fonte: API de Consulta do PNCP, endpoint "GET /v1/contratacoes/publicacao"
([Swagger](https://pncp.gov.br/api/consulta/swagger-ui/index.html)).

Feito nesta camada:
1. Percorre as modalidades, mês a mês e página a página para a UF escolhida.
2. Salva os registros num volume do Unity Catalog.
3. Cria a tabela Delta "bronze.pncp_contratacoes" com metadados de ingestão.

In [0]:

dbutils.widgets.text("uf", "PA", "UF")
dbutils.widgets.text("data_inicial", "20260101", "Data inicial (yyyyMMdd)")
dbutils.widgets.text("data_final", "20260831", "Data final (yyyyMMdd)")

UF = dbutils.widgets.get("uf").strip().upper()
DATA_INICIAL = dbutils.widgets.get("data_inicial").strip()
DATA_FINAL = dbutils.widgets.get("data_final").strip()

BASE_URL = "https://pncp.gov.br/api/consulta/v1/contratacoes/publicacao"
TAMANHO_PAGINA = 50 

MODALIDADES = {
    1: "Leilão - Eletrônico",
    2: "Diálogo Competitivo",
    3: "Concurso",
    4: "Concorrência - Eletrônica",
    5: "Concorrência - Presencial",
    6: "Pregão - Eletrônico",
    7: "Pregão - Presencial",
    8: "Dispensa de Licitação",
    9: "Inexigibilidade",
    10: "Manifestação de Interesse",
    11: "Pré-qualificação",
    12: "Credenciamento",
    13: "Leilão - Presencial",
}

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE VOLUME IF NOT EXISTS workspace.bronze.raw_pncp;

In [0]:
import calendar
import json
import os
import time
from datetime import date, datetime, timezone

import requests

sessao = requests.Session()
sessao.headers.update({"accept": "application/json"})


def janelas_mensais(inicio: str, fim: str):
    """Divide o período em janelas de um mês (consultas menores e mais estáveis)."""
    atual = datetime.strptime(inicio, "%Y%m%d").date()
    limite = datetime.strptime(fim, "%Y%m%d").date()
    while atual <= limite:
        ultimo_dia = date(atual.year, atual.month, calendar.monthrange(atual.year, atual.month)[1])
        yield atual, min(ultimo_dia, limite)
        atual = date(atual.year + atual.month // 12, atual.month % 12 + 1, 1)


def buscar_pagina(params: dict, tentativas: int = 5) -> dict:
    """Faz a requisição com nova tentativa (backoff exponencial) para erros temporários."""
    for t in range(tentativas):
        try:
            r = sessao.get(BASE_URL, params=params, timeout=60)
        except requests.RequestException as e:
            print(f"    erro de rede ({e}); nova tentativa em {2 ** t}s")
            time.sleep(2 ** t)
            continue

        if r.status_code == 200:
            return r.json()
        if r.status_code == 204:  
            return {"data": [], "totalPaginas": 0}
        if r.status_code in (429, 500, 502, 503, 504):
            print(f"    HTTP {r.status_code}; nova tentativa em {5 ** t}s")
            time.sleep(5 ** t)
            continue
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:300]}")
    raise RuntimeError(f"falhou após {tentativas} tentativas")


def extrair_janela(modalidade: int, ini: date, fim: date) -> list:
    """Percorre todas as páginas de uma modalidade em uma janela de datas."""
    registros, pagina = [], 1
    while True:
        resp = buscar_pagina({
            "dataInicial": ini.strftime("%Y%m%d"),
            "dataFinal": fim.strftime("%Y%m%d"),
            "codigoModalidadeContratacao": modalidade,
            "uf": UF,
            "pagina": pagina,
            "tamanhoPagina": TAMANHO_PAGINA,
        })
        dados = resp.get("data") or []
        registros.extend(dados)

        total_paginas = resp.get("totalPaginas")
        acabou = (
            not dados
            or (total_paginas is not None and pagina >= total_paginas)
            or (total_paginas is None and len(dados) < TAMANHO_PAGINA)
        )
        if acabou:
            return registros
        pagina += 1
        time.sleep(0.2) 

In [0]:
execucao_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
destino = f"/Volumes/workspace/bronze/raw_pncp/contratacoes/execucao={execucao_id}"
os.makedirs(destino, exist_ok=True)

log, falhas = [], []
inicio_execucao = time.time()

for cod, nome in MODALIDADES.items():
    for ini, fim in janelas_mensais(DATA_INICIAL, DATA_FINAL):
        try:
            registros = extrair_janela(cod, ini, fim)
        except RuntimeError as e:
            falhas.append((cod, f"{ini:%Y-%m}", str(e)))
            print(f"[FALHA] {cod:>2} {nome} {ini:%Y-%m}: {e}")
            continue

        if registros:
            arquivo = f"{destino}/mod{cod:02d}_{ini:%Y%m}.jsonl"
            with open(arquivo, "w", encoding="utf-8") as f:
                for reg in registros:
                    f.write(json.dumps(reg, ensure_ascii=False) + "\n")

        log.append((cod, nome, f"{ini:%Y-%m}", len(registros)))
        print(f"{cod:>2} {nome:<28} {ini:%Y-%m}: {len(registros):>6} registros")

total = sum(qtd for *_, qtd in log)
print(f"\nTotal: {total} registros em {(time.time() - inicio_execucao) / 60:.1f} min | falhas: {len(falhas)}")

 1 Leilão - Eletrônico          2026-01:      1 registros
 1 Leilão - Eletrônico          2026-02:      1 registros
 1 Leilão - Eletrônico          2026-03:      0 registros
 1 Leilão - Eletrônico          2026-04:      0 registros
 1 Leilão - Eletrônico          2026-05:      1 registros
 1 Leilão - Eletrônico          2026-06:      1 registros
 1 Leilão - Eletrônico          2026-07:      2 registros
 1 Leilão - Eletrônico          2026-08:      1 registros
 2 Diálogo Competitivo          2026-01:      0 registros
    HTTP 429; nova tentativa em 1s
    HTTP 429; nova tentativa em 5s
    HTTP 429; nova tentativa em 25s
 2 Diálogo Competitivo          2026-02:      0 registros
 2 Diálogo Competitivo          2026-03:      0 registros
 2 Diálogo Competitivo          2026-04:      0 registros
 2 Diálogo Competitivo          2026-05:      0 registros
 2 Diálogo Competitivo          2026-06:      0 registros
 2 Diálogo Competitivo          2026-07:      0 registros
 2 Diálogo Competitivo  

In [0]:
from pyspark.sql import functions as F

df_bronze = (
    spark.read.json(destino)
    .withColumn("_arquivo_origem", F.col("_metadata.file_path"))
    .withColumn("_data_ingestao", F.current_timestamp())
    .withColumn("_fonte", F.lit(BASE_URL))
    .withColumn("_execucao_id", F.lit(execucao_id))
)

(
    df_bronze.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.pncp_contratacoes")
)

spark.sql(f"""
COMMENT ON TABLE workspace.bronze.pncp_contratacoes IS
'Camada Bronze: contratações publicadas no PNCP (UF={UF}, {DATA_INICIAL} a {DATA_FINAL}), todas as modalidades. \
Registros sem transformação, extraídos de {BASE_URL}. Colunas iniciadas com _ são metadados de ingestão.'
""")

DataFrame[]

In [0]:
df_log = spark.createDataFrame(log, "codigo INT, modalidade STRING, mes STRING, registros INT")
display(df_log.groupBy("codigo", "modalidade").agg(F.sum("registros").alias("registros")).orderBy("codigo"))

codigo,modalidade,registros
1,Leilão - Eletrônico,7
2,Diálogo Competitivo,0
3,Concurso,5
4,Concorrência - Eletrônica,1487
5,Concorrência - Presencial,29
6,Pregão - Eletrônico,5566
7,Pregão - Presencial,100
8,Dispensa de Licitação,6438
9,Inexigibilidade,3534
10,Manifestação de Interesse,0


In [0]:
display(spark.sql(f"SELECT COUNT(*) AS total_registros FROM workspace.bronze.pncp_contratacoes"))

total_registros
17432


In [0]:
display(spark.table("workspace.bronze.pncp_contratacoes").limit(10))

amparoLegal,anoCompra,dataAberturaProposta,dataAtualizacao,dataAtualizacaoGlobal,dataEncerramentoProposta,dataInclusao,dataPublicacaoPncp,emendaParlamentar,fontesOrcamentarias,informacaoComplementar,justificativaPresencial,linkProcessoEletronico,linkSistemaOrigem,modalidadeId,modalidadeNome,modoDisputaId,modoDisputaNome,numeroCompra,numeroControlePNCP,objetoCompra,orgaoEntidade,orgaoSubRogado,processo,sequencialCompra,situacaoCompraId,situacaoCompraNome,srp,tipoInstrumentoConvocatorioCodigo,tipoInstrumentoConvocatorioNome,unidadeOrgao,unidadeSubRogada,usuarioNome,valorTotalEstimado,valorTotalHomologado,_arquivo_origem,_data_ingestao,_fonte,_execucao_id
"List(4, Leilão: modalidade de licitação para alienação de bens imóveis ou de bens móveis inservíveis ou legalmente apreendidos a quem oferecer o maior lance, Lei 14.133/2021, Art. 28, IV)",2026,2026-05-07T00:00:00,2026-05-06T16:07:31,2026-05-06T16:07:33,2026-05-28T00:00:00,2026-01-13T09:53:30,2026-01-13T09:53:30,null,List(),,"O presente expediente tem por finalidade justificar a alienação do bem imóvel pertencente ao patrimônio do CRBM-4, considerando critérios de economicidade, eficiência administrativa, segurança institucional e interesse público, nos termos que regem a Administração Pública.",null,,1,Leilão - Eletrônico,1,Aberto,2026.001-00-CRBM-4,34639419000100-1-000001/2026,"Constitui objeto deste Leilão, a alienação de BEM IMÓVEL de propriedade do Conselho Regional de Biomedicina - 4ª Região, relacionado e avaliado no Anexo I do edital, no qual constam informações sobre a descrição do bem, o valor de avaliação e preço mínimo de arrematação, localização e respectivos períodos, horários, datas e contatos para visitação.","List(34639419000100, F, E, CONSELHO REGIONAL DE BIOMEDICINA - 4A REGIAO)",null,2026.001-00-CRBM-4,1,1,Divulgada no PNCP,false,1,Edital,"List(1501402, 389112, Belém, CONSELHO REGIONAL DE BIOMEDICINA/PA, Pará, PA)",null,Implanta Informatica LTDA,742013.56,null,dbfs:/Volumes/workspace/bronze/raw_pncp/contratacoes/execucao=20260920T005732/mod01_202601.jsonl,2026-09-20T01:35:43.901Z,https://pncp.gov.br/api/consulta/v1/contratacoes/publicacao,20260920T005732
"List(4, Leilão: modalidade de licitação para alienação de bens imóveis ou de bens móveis inservíveis ou legalmente apreendidos a quem oferecer o maior lance, Lei 14.133/2021, Art. 28, IV)",2026,2026-03-12T10:00:00,2026-02-26T11:00:01,2026-02-26T11:03:01,2026-03-12T12:00:00,2026-02-26T11:00:01,2026-02-26T11:00:01,null,List(),"O bem deverá ser visitado pelos interessados nos dias 09 a 11 de março de 2026, no horário de 08 às 14h, no endereço Av. João Paulo II, Bairro Curió-Utinga, s/nº, Belém-PA, no Batalhão de Polícia Ambiental da PM-PA, mediante agendamento prévio junto à Polícia Militar do Pará, por meio do contato telefônico (91) 98479-7136, informando nome completo, número de RG e CPF. Para acesso ao local de guarda da embarcação, será exigida do interessado a apresentação de documento com foto",null,null,,1,Leilão - Eletrônico,1,Aberto,L 01,35747782000101-1-000007/2026,"O objeto do presente Leilão é a venda da embarcação tipo lancha, denominada MOREIA, oriunda de Órgão da Administração Pública Estadual","List(35747782000101, E, E, SECRETARIA DE ESTADO DE PLANEJAMENTO E ADMINISTRACAO)",null,2139916,7,1,Divulgada no PNCP,false,1,Edital,"List(1501402, 1, Belém, SEPLAD, Pará, PA)",null,Compras Pará,200000.0,null,dbfs:/Volumes/workspace/bronze/raw_pncp/contratacoes/execucao=20260920T005732/mod01_202602.jsonl,2026-09-20T01:35:43.901Z,https://pncp.gov.br/api/consulta/v1/contratacoes/publicacao,20260920T005732
"List(4, Leilão: modalidade de licitação para alienação de bens imóveis ou de bens móveis inservíveis ou legalmente apreendidos a quem oferecer o maior lance, Lei 14.133/2021, Art. 28, IV)",2026,2026-06-22T08:00:00,2026-05-25T09:33:55,2026-05-25T09:33:56,2026-07-06T20:00:00,2026-05-25T09:33:55,2026-05-25T09:33:55,null,List(),"Todos os detalhes do leilão poderão ser visualizados no site , inclusive eventua

In [0]:
if falhas:
    display(spark.createDataFrame(falhas, "codigo INT, mes STRING, erro STRING"))
else:
    print("Nenhuma falha na extração.")

Nenhuma falha na extração.
